In [2]:
import pandas as pd
import glob, os

path = "Raw-data/"   # لو الـ Notebook جوه olist-project مباشرة
tables = {}
for f in glob.glob(path + "*.csv"):
    name = os.path.basename(f).replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
    tables[name] = pd.read_csv(f)

for name, df in tables.items():
    print(f"{name:35s} rows={df.shape[0]:>7,}  cols={df.shape[1]}")

customers                           rows= 99,441  cols=5
geolocation                         rows=1,000,163  cols=5
orders                              rows= 99,441  cols=8
order_items                         rows=112,650  cols=7
order_payments                      rows=103,886  cols=5
order_reviews                       rows= 99,224  cols=7
products                            rows= 32,951  cols=9
sellers                             rows=  3,095  cols=4
product_category_name_translation   rows=     71  cols=2


In [3]:
for name, df in tables.items():
    print(f"\n=== {name} ===")
    print("Duplicates:", df.duplicated().sum())
    nulls = df.isna().sum()
    print(nulls[nulls > 0] if nulls.sum() > 0 else "No nulls")


=== customers ===
Duplicates: 0
No nulls

=== geolocation ===
Duplicates: 261831
No nulls

=== orders ===
Duplicates: 0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

=== order_items ===
Duplicates: 0
No nulls

=== order_payments ===
Duplicates: 0
No nulls

=== order_reviews ===
Duplicates: 0
review_comment_title      87656
review_comment_message    58247
dtype: int64

=== products ===
Duplicates: 0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

=== sellers ===
Duplicates: 0
No nulls

=== product_category_name_translation ===
Duplicates: 0
No nulls


In [4]:
orders = tables["orders"]
items = tables["order_items"]
reviews = tables["order_reviews"]

print("Unique orders:", orders["order_id"].nunique(), "of", len(orders))
print("Orders in items:", items["order_id"].nunique())
print("Orders in reviews:", reviews["order_id"].nunique(), "of", len(reviews))
print("\nOrder status:")
print(orders["order_status"].value_counts())

Unique orders: 99441 of 99441
Orders in items: 98666
Orders in reviews: 98673 of 99224

Order status:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [5]:
no_items = orders[~orders["order_id"].isin(items["order_id"])]

print(len(no_items))
print(no_items["order_status"].value_counts())

775
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


In [6]:
no_reviews = orders[~orders["order_id"].isin(reviews["order_id"])]

print(len(no_reviews))
print(no_reviews["order_status"].value_counts())

768
order_status
delivered      646
shipped         75
canceled        20
unavailable     14
processing       6
invoiced         5
created          2
Name: count, dtype: int64


In [7]:
reviews.columns

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')

In [8]:
counts = reviews["order_id"].value_counts()
dup_orders = counts[counts > 1]

print(len(dup_orders))
print(counts.max())
print((dup_orders - 1).sum())
print(dup_orders.value_counts())

547
3
551
count
2    543
3      4
Name: count, dtype: int64


In [9]:
reviews_clean = (
    reviews
    .sort_values(["order_id", "review_answer_timestamp"])
    .drop_duplicates(subset="order_id", keep="last")
)

print(len(reviews_clean))
print(reviews_clean["order_id"].is_unique)

98673
True


In [10]:
dups = reviews[reviews["order_id"].isin(dup_orders.index)]
score_diff = dups.groupby("order_id")["review_score"].nunique()
print((score_diff > 1).sum())

202


In [11]:
odd = orders[(orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"].isna())]

print(len(odd))
odd

8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00
